# ProteinGym vs UKBBGym: signal-to-noise of predictor deltas, percentile-bootstrap CI

Stripped-down remake of [`protein_gym_delta_mean.ipynb`](protein_gym_delta_mean.ipynb): the same
analysis, cut down to the two figures that matter — **overall SNR** and **SNR per readout** —
with the SEM replaced by a percentile-bootstrap confidence interval. Everything else in that
notebook (assay-count bar charts, region-overlap checks, the Mammalian-vs-Primate PhyloP deep
dive, the parallel assay-level/gene-level pair of every plot) is gone.

For each dataset independently, the estimand for a pair of predictors (A, B) is the mean over
protein-units of the **paired** per-unit delta `corr_A(unit) − corr_B(unit)`. Pairing matters: A
and B are scored on the same units, and one index draw per replicate is shared across every
pair, so the unit-level noise A and B have in common cancels in the delta instead of being
counted twice. Signal-to-noise is then

$$\mathrm{SNR} \;=\; \frac{\hat\Delta}{\text{CI}_{hi} - \text{CI}_{lo}}$$

with the bounds read straight off the bootstrap distribution as its α/2 and 1−α/2 quantiles.
Significance is **not** a threshold on SNR — it is `is_significant`, read from the bounds
themselves (`ci_lo > 0` or `ci_hi < 0`), so an asymmetric interval is never forced through a
symmetric decision rule. (For calibration only: if a CI happened to be symmetric,
`hi − lo = 3.92·SE`, so SNR ≈ 0.26 would be where it just excludes zero. Do not read that number
off the plots — the whole point is that these intervals are not symmetric.)

### Why the percentile interval, and why nothing more clever

`mean ± 1.96·SEM` is a 95% interval only if the sampling distribution of Δ̂ is Gaussian. With 480
genes (UKBBGym) it is. With the 7 assays in the `Binding` stratum it is not, and the per-unit
deltas are not mild — they reach skewness −4.6 and excess kurtosis 32, because one assay that
disagrees with the rest dominates the mean.

The percentile interval takes quantiles of Δ̂\* directly, so **whatever shape the bootstrap
distribution has, the interval inherits it as-is** — skewed, long-tailed, lopsided about the
point estimate. Nothing is assumed symmetric and nothing is corrected. That skew is real
information about what a small n is doing to the estimate, so it is *reported* (`boot_skew`, and
the arm ratio `(hi − Δ̂)/(Δ̂ − lo)`), never used to adjust the bounds.

That rules out BCa, which shifts the same quantiles by two data-estimated corrections — z₀ for
median bias and *a* for skewness, the latter a jackknife estimate. Over 7–26 units that jackknife
is as uncertain as the thing it corrects, and in the coverage simulation in
[`protein_gym_delta_snr_ci.ipynb`](protein_gym_delta_snr_ci.ipynb) BCa came out **worst** of five
methods (0.81 coverage at every n on the heavy-tailed pair). First-order-and-honest beats
second-order-and-fragile here.

**The cost, stated plainly:** the percentile interval is honest about shape but optimistic about
width. In that same simulation it held 0.85–0.89 coverage at n = 7 and 0.81–0.89 on the most
skewed pair — better than BCa, worse than a sign-flip randomisation interval, which reached
0.94–0.96 everywhere. So `is_significant` in the small strata is a somewhat liberal flag. That is
a known, bounded, deliberate trade for an interval with no second estimate that can misfire; the
sign-flip alternative is one function call away in the other notebook if a specific pair needs
checking.

UKBBGym's protein-unit is the gene. ProteinGym's is set by `UNIT_LEVEL`:

- `'gene'` (default) — units are **genes**, each contributing all of its assays. 57 genes supply
  70 assays, and two assays of the same gene are not independent evidence about a predictor, so
  treating assays as units would overweight multi-assay genes and understate the uncertainty.
- `'assay'` — units are assays, matching the older half of the parent notebook.

In [ ]:
import yaml
import numpy as np
import polars as pl
from scipy import stats

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

## Parameters

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'variant_filtering.py').exists())
sys.path.insert(0, str(REPO_ROOT))
from utils.variant_filtering import env_override, fetch_hf_data

# --- parameters (env_override(NAME, default) -- set UKBBGYM_NAME to override without editing) ---
variant_class       = env_override('VARIANT_CLASS', 'missense')
config_file         = env_override('CONFIG_FILE', 'config_correlations.yaml')

selected_categories = env_override('SELECTED_CATEGORIES',
                                    ['missense', 'conservation', 'genetic_diversity', 'gnomad'], 'list')
mac                 = env_override('MAC', 20, int)
only_snps           = env_override('ONLY_SNPS', True, bool)
MIN_VARIANTS        = env_override('MIN_VARIANTS', 100, int)     # protein-units below this variant count are dropped
CONFIDENCE          = env_override('CONFIDENCE', 0.95, float)    # percentile-bootstrap interval level
B                   = env_override('N_BOOT', 10000, int)   # bootstrap replicates; tail quantiles need more than a mean does
SEED                = env_override('SEED', 0, int)
UNIT_LEVEL          = env_override('UNIT_LEVEL', 'gene')  # ProteinGym resampling unit: 'gene' (cluster) or 'assay'

MASTER_PATH     = env_override('MASTER_PATH', fetch_hf_data('genebass_annotated.parquet', REPO_ROOT))
PROTEINGYM_FILE = env_override('PROTEINGYM_PATH', fetch_hf_data('other_benchmarks/proteingym_snv_annotated.parquet', REPO_ROOT))
CONFIG_DIR      = str(REPO_ROOT / 'configs')
FIG_DIR         = env_override('FIG_DIR', '../../../paper_figures')

## Load

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT))
from utils.variant_filtering import (
    load_config, load_variant_class, scan_variants, appv_of, filter_covered,
    build_gene_trait_tool_correlations,
)

anno_config_df, all_annotation_list = load_config(CONFIG_DIR, config_file)
vc = load_variant_class(CONFIG_DIR, variant_class)
anno_sel = anno_config_df.filter(pl.col('category').is_in(selected_categories))
print(f'{len(all_annotation_list)} annotations in config; {anno_sel.height} in selected categories')


## Per-protein Spearman correlations

ProteinGym: one correlation per (DMS assay, predictor), predictor score against `dms_score`.
UKBBGym: one per (gene, predictor), predictor score against the per-variant Genebass beta.
Both are direction-corrected so a positive value always means "higher score → more damaging".

In [ ]:
pgdf = pl.read_parquet(PROTEINGYM_FILE)
existing_pg_annos = [c for c in all_annotation_list if c in pgdf.columns]

# gnomAD AF is null when a variant simply isn't in gnomAD -- closer to frequency 0 than to
# missing data. Left null, those rows fail the "every unit has every annotation" coverage
# filter below and get dropped wholesale; filling with 0 keeps them instead.
gnomad_annos = anno_sel.filter(pl.col('category') == 'gnomad')['annotation'].to_list()

pg_corr = (
    pgdf.select(list({'file_name', 'exp_readout', 'id', 'region', 'dms_score'} | set(existing_pg_annos)))
    .unpivot(index=['file_name', 'exp_readout', 'id', 'region', 'dms_score'],
             on=existing_pg_annos, variable_name='annotation', value_name='annotation_score')
    .with_columns(pl.when(pl.col('annotation').is_in(gnomad_annos))
                    .then(pl.col('annotation_score').fill_null(0))
                    .otherwise(pl.col('annotation_score'))
                    .alias('annotation_score'))
    .with_columns(pl.col(c).rank('average').over(['region', 'file_name', 'exp_readout', 'annotation']).alias(f'{c}_rank')
                  for c in ['annotation_score', 'dms_score'])
    .group_by(['region', 'file_name', 'exp_readout', 'annotation'])
    .agg(n_variants=pl.col('id').count(),
         correlation=pl.when((pl.col('annotation_score_rank').n_unique() > 1) &
                             (pl.col('dms_score_rank').n_unique() > 1))
                       .then(pl.corr('annotation_score_rank', 'dms_score_rank', propagate_nans=True))
                       .otherwise(None))
    .drop_nans().drop_nulls()
    .join(anno_sel, on='annotation')
    .with_columns(corr_dircor=pl.col('correlation') * pl.col('annotation_dir') * -1)
)

# file_name must be a unique protein-unit key, or the pivot below silently collapses rows.
n_bad = (pg_corr.group_by('file_name').agg(pl.col('exp_readout').n_unique().alias('n'))['n'] > 1).sum()
if n_bad:
    print(f'{n_bad} file_name values span multiple exp_readouts -- switching to a composite key')
    pg_corr = pg_corr.with_columns((pl.col('file_name') + '::' + pl.col('exp_readout')).alias('file_name'))

ukbb_lf = scan_variants(MASTER_PATH, vc, only_snps=only_snps)
ukbb_names = ukbb_lf.collect_schema().names()
existing_ukbb_annos = [a for a in anno_sel['annotation'].to_list() if a in ukbb_names]

# Same gnomAD null->0 fill as the ProteinGym side above, applied to the raw columns before the
# shared correlation computation melts them (equivalent to filling the melted annotation_score
# afterward, since unpivot doesn't touch values).
existing_gnomad_annos = [a for a in gnomad_annos if a in ukbb_names]
if existing_gnomad_annos:
    ukbb_lf = ukbb_lf.with_columns([pl.col(c).fill_null(0) for c in existing_gnomad_annos])

# Same canonical per-(gene, trait, tool) correlation computation as correlations.ipynb
# / clinvar_spearman_scatterplot.ipynb / ac_robustness_master_file.ipynb -- coverage
# filter deliberately NOT applied yet (build_gene_trait_tool_correlations, not
# gene_trait_tool_correlations): the threshold here is common_annos, the tool set actually shared
# with ProteinGym, which isn't known until the next cell.
ukbb_corr = build_gene_trait_tool_correlations(ukbb_lf, mac, existing_ukbb_annos, anno_config_df,
                                               selected_categories)
print(f'ProteinGym {pg_corr.shape}, UKBBGym {ukbb_corr.shape}')

## Common predictors, coverage filter, wide matrices

Keep only predictors present in both datasets, then only protein-units that clear
`MIN_VARIANTS` **and** carry a correlation for every one of them — so the matrices below have no
missing cells and the bootstrap needs no NaN handling.

In [ ]:
common_annos = sorted(set(pg_corr['annotation'].unique()) & set(ukbb_corr['annotation'].unique()))
K = len(common_annos)
print(f'{K} predictors common to both datasets: {common_annos}')


pg_filt = filter_covered(pg_corr.filter(pl.col('annotation').is_in(common_annos)),
                         group_col='file_name', n_variants_col='n_variants', min_variants=MIN_VARIANTS)
ukbb_filt = filter_covered(ukbb_corr.filter(pl.col('annotation').is_in(common_annos)),
                           group_col='region', n_variants_col='n_variants', min_variants=MIN_VARIANTS)

# .sort() is not cosmetic: group_by/pivot row order is not deterministic, and unit order is what
# the sign patterns land on -- without it every re-run randomises slightly differently.
pg_wide = (pg_filt.pivot(index=['file_name', 'exp_readout', 'region'], on='annotation', values='corr_dircor')
           .sort('file_name'))
ukbb_wide = ukbb_filt.pivot(index='region', on='annotation', values='corr_beta').sort('region')
assert pg_wide.select(common_annos).null_count().sum_horizontal().sum() == 0
assert ukbb_wide.select(common_annos).null_count().sum_horizontal().sum() == 0

pg_scores, ukbb_scores = pg_wide.select(common_annos).to_numpy(), ukbb_wide.select(common_annos).to_numpy()
pg_readout, pg_gene_ids = pg_wide['exp_readout'].to_numpy(), pg_wide['region'].to_numpy()
n_pg, n_ukbb, n_genes_pg = len(pg_scores), len(ukbb_scores), pg_wide['region'].n_unique()
print(f'ProteinGym: {pg_corr["file_name"].n_unique()} assays -> {n_pg} after filtering, across {n_genes_pg} genes')
print(f'UKBBGym:    {ukbb_corr["region"].n_unique()} genes  -> {n_ukbb} after filtering')

## Per-unit deltas, then the percentile bootstrap

`compute_pairwise_delta_matrix` builds `D`, the per-unit paired delta for every predictor pair —
shape `(n_units, K, K)`, nothing bootstrapped yet. It also returns `den`, the number of rows each
unit contributes, which is what lets one code path serve both resampling levels:

    Δ̂ = sum_i D_i / sum_i den_i

With `den = 1` (units are assays) that is literally the mean of the per-unit deltas. With `den` =
the gene's assay count it is the ratio estimator behind the cluster bootstrap — the mean over all
assays of the drawn genes, without ever building those rows.

`bootstrap_percentile_snr` then draws `B` sets of `n` indices with replacement from **one** RNG
stream and applies each draw to every predictor pair at once, which is what preserves the paired
cancellation. The draws are represented as multinomial counts rather than an explicit
`(B, n, K, K)` gather — distributionally identical, but it keeps the memory at `(B, n)` instead
of 55 GB.

The point estimate is the **unbootstrapped** mean over the original units, so no bootstrap bias
enters the numerator. Everything is vectorised over the full (K × K) grid: all 66 pairs in one
pass.

In [ ]:
def compute_pairwise_delta_matrix(scores, cluster_ids=None):
    '''Per-unit paired deltas D[i, a, b] = scores[i, a] - scores[i, b], plus each unit's weight.

    cluster_ids=None -> one row per unit, den == 1, and D is exactly the per-unit delta.
    cluster_ids given -> units are clusters (genes): D[g] is the cluster's summed deltas and
    den[g] its row count, so a drawn gene contributes all of its assays.
    Returns D (n_units, K, K), den (n_units,).'''
    if cluster_ids is not None:
        ug, gi = np.unique(cluster_ids, return_inverse=True)
        summed = np.zeros((len(ug), scores.shape[1]))
        np.add.at(summed, gi, scores)
        scores, den = summed, np.bincount(gi, minlength=len(ug)).astype(float)
    else:
        den = np.ones(scores.shape[0])
    return scores[:, :, None] - scores[:, None, :], den


def bootstrap_percentile_snr(D, den, B=B, seed=SEED, confidence=CONFIDENCE):
    '''Percentile-bootstrap CI and SNR for every predictor pair.

    No symmetry assumption and no bias/skewness correction: the bounds are the raw quantiles of
    the bootstrap distribution, so whatever shape it has -- skewed, long-tailed, lopsided about
    the point estimate -- the interval inherits it. Skewness is returned as a diagnostic, not
    used to adjust anything.'''
    n, Kc = D.shape[0], D.shape[1]
    alpha = 1 - confidence
    # One RNG stream, one draw per replicate, applied to every pair at once -- multinomial counts
    # are exactly resampling n indices with replacement, without an (B, n, K, K) gather.
    counts = np.random.default_rng(seed).multinomial(n, np.full(n, 1 / n), size=B).astype(np.float64)
    boot_D = ((counts @ D.reshape(n, -1)) / (counts @ den)[:, None]).reshape(B, Kc, Kc)

    obs_mean = D.sum(0) / den.sum()                       # point estimate, unbootstrapped
    ci_lo, ci_hi = np.quantile(boot_D, [alpha / 2, 1 - alpha / 2], axis=0)
    ci_width = ci_hi - ci_lo
    snr = np.divide(obs_mean, ci_width, out=np.full_like(obs_mean, np.nan), where=ci_width > 0)
    is_significant = (ci_lo > 0) | (ci_hi < 0)

    boot_skew = stats.skew(boot_D, axis=0)                # reported, never corrected for
    lo_arm, hi_arm = obs_mean - ci_lo, ci_hi - obs_mean
    arm_ratio = np.divide(hi_arm, lo_arm, out=np.full_like(obs_mean, np.nan), where=lo_arm > 0)
    return dict(obs_mean=obs_mean, ci_lo=ci_lo, ci_hi=ci_hi, ci_width=ci_width, snr=snr,
                is_significant=is_significant, boot_skew=boot_skew, arm_ratio=arm_ratio,
                n_units=n)


gids = pg_gene_ids if UNIT_LEVEL == 'gene' else None
n_units_pg = n_genes_pg if UNIT_LEVEL == 'gene' else n_pg

ukbb_res = bootstrap_percentile_snr(*compute_pairwise_delta_matrix(ukbb_scores), seed=SEED)
pg_res_overall = bootstrap_percentile_snr(*compute_pairwise_delta_matrix(pg_scores, gids), seed=SEED + 500)

pg_res_by_readout = {}
for i, readout in enumerate(sorted(set(pg_readout))):
    m = pg_readout == readout
    n_g = len(np.unique(pg_gene_ids[m]))
    pg_res_by_readout[readout] = (bootstrap_percentile_snr(
        *compute_pairwise_delta_matrix(pg_scores[m], None if gids is None else gids[m]),
        seed=SEED + 600 + i), int(m.sum()), n_g)
    print(f'{readout}: {m.sum()} assays across {n_g} genes')
print(f'\npercentile bootstrap done ({UNIT_LEVEL}-level, {B} replicates, '
      f'{int(100 * CONFIDENCE)}% CI): ProteinGym {n_units_pg} units, UKBBGym {n_ukbb} genes')

## Flatten to one row per predictor pair

In [ ]:
label_map = dict(anno_config_df.select('annotation', 'label').unique().iter_rows())
cat_map = dict(anno_config_df.select('annotation', 'category').unique().iter_rows())
cat_color = dict(anno_config_df.filter(pl.col('annotation').is_in(common_annos))
                 .group_by('category').agg(pl.col('color').first()).iter_rows())
iu, ju = np.triu_indices(K, k=1)


FIELDS = ['obs_mean', 'ci_lo', 'ci_hi', 'ci_width', 'snr', 'is_significant', 'boot_skew', 'arm_ratio']


def to_pairs(res, prefix):
    return pl.DataFrame({'predictor_a': [common_annos[i] for i in iu],
                         'predictor_b': [common_annos[j] for j in ju],
                         **{f'{prefix}_{f}': res[f][iu, ju] for f in FIELDS}})



SIG5_CATEGORIES = ['Both, signs agree', 'Both, signs disagree', 'Neither', 'ProteinGym only', 'UKBBGym only']


def decorate(df):
    both_sig = pl.col('pg_is_significant') & pl.col('ukbb_is_significant')
    signs_agree = pl.col('pg_snr').sign() == pl.col('ukbb_snr').sign()
    return df.with_columns(pair_label=pl.col('predictor_a').replace(label_map) + ' vs ' +
                                      pl.col('predictor_b').replace(label_map),
                           category_a=pl.col('predictor_a').replace(cat_map),
                           significance=pl.when(pl.col('pg_is_significant') & pl.col('ukbb_is_significant'))
                                          .then(pl.lit('Both'))
                                          .when(pl.col('pg_is_significant')).then(pl.lit('ProteinGym only'))
                                          .when(pl.col('ukbb_is_significant')).then(pl.lit('UKBBGym only'))
                                          .otherwise(pl.lit('Neither'))
                                          .cast(pl.Enum(['Both', 'ProteinGym only', 'UKBBGym only', 'Neither'])),
                           # Finer than `significance`: 'both' splits into signs-agree / signs-disagree,
                           # since sign agreement is only meaningful once both datasets resolve the pair.
                           significance5=pl.when(both_sig & signs_agree).then(pl.lit('Both, signs agree'))
                                           .when(both_sig).then(pl.lit('Both, signs disagree'))
                                           .when(pl.col('pg_is_significant')).then(pl.lit('ProteinGym only'))
                                           .when(pl.col('ukbb_is_significant')).then(pl.lit('UKBBGym only'))
                                           .otherwise(pl.lit('Neither'))
                                           .cast(pl.Enum(SIG5_CATEGORIES)))


ukbb_pairs = to_pairs(ukbb_res, 'ukbb')
pairs_overall = decorate(to_pairs(pg_res_overall, 'pg').join(ukbb_pairs, on=['predictor_a', 'predictor_b']))

pairs_readout = decorate(
    pl.concat([to_pairs(res, 'pg').with_columns(exp_readout=pl.lit(readout),
                                                pg_n_assays=n_a, pg_n_genes=n_g)
               for readout, (res, n_a, n_g) in pg_res_by_readout.items()])
    .join(ukbb_pairs, on=['predictor_a', 'predictor_b'])
).with_columns(readout_label=pl.col('exp_readout') + ' (' +
                             (pl.col('pg_n_genes') if UNIT_LEVEL == 'gene'
                              else pl.col('pg_n_assays')).cast(pl.Utf8) + f' {UNIT_LEVEL}s)')

print(f'{pairs_overall.height} predictor pairs; '
      f'{pairs_overall.height} x {len(pg_res_by_readout)} = {pairs_readout.height} rows stratified')
print(f'significant (CI excludes 0) -- ProteinGym overall: {pairs_overall["pg_is_significant"].sum()}, '
      f'UKBBGym: {pairs_overall["ukbb_is_significant"].sum()}')
print(pairs_overall['significance'].value_counts().sort('significance'))
pairs_overall.head()

## Plot 1 — overall signal-to-noise

Marker shape *and* fill both carry `is_significant` — whether that dataset's CI excludes zero —
read from the bounds themselves rather than from a cutoff on SNR, since with asymmetric intervals
no single SNR cutoff corresponds to "excludes zero". Encoding it twice keeps the four states
separable in greyscale and at small print sizes; the two scales share a `name`, which is what
makes plotnine merge them into a single legend. Dark filled circles are pairs both datasets call
significant, the set worth believing; hollow diamonds are pairs neither can resolve.

One deliberate change from the parent notebook: ρ is the Spearman correlation of the **signed**
SNRs, the two quantities actually on the axes. The parent plots signed SNR but annotates ρ
computed on `|SNR|`, which answers a different question (do the datasets agree on how *confident*
each comparison is) than the plot asks (do they agree on which predictor wins, and by how much).

In [ ]:
unit_word = 'genes' if UNIT_LEVEL == 'gene' else 'assays'
ci_axis = f'Δcorr / width of the {int(100 * CONFIDENCE)}%\npercentile-bootstrap CI'

SIG_LEGEND = 'CI excludes 0 in'          # must be identical in both scales to merge them
shape_vals = {'Both': 'o', 'ProteinGym only': '^', 'UKBBGym only': 's', 'Neither': 'D'}
fill_vals  = {'Both': '#1B4965', 'ProteinGym only': '#5FA8D3',
              'UKBBGym only': '#F4A259', 'Neither': 'white'}

rho_overall = pairs_overall.select(
    spearman_r=pl.corr(pl.col('pg_snr').rank('average'), pl.col('ukbb_snr').rank('average')),
    x=pl.col('pg_snr').min(), y=pl.col('ukbb_snr').max(),
).with_columns(corr_label='ρ = ' + pl.col('spearman_r').round(2).cast(pl.Utf8))

plot_overall = (
    ggplot(pairs_overall, aes(x='pg_snr', y='ukbb_snr'))
    + geom_hline(yintercept=0, color='lightgrey')
    + geom_vline(xintercept=0, color='lightgrey')
    + geom_abline(slope=1, intercept=0, linetype='dashed', color='grey')
    + geom_point(aes(shape='significance', fill='significance'), color='black', size=2.8,
                 stroke=0.4, alpha=0.85)
    + geom_text(data=rho_overall, mapping=aes(x='x', y='y', label='corr_label'),
                ha='left', va='top', size=12, fontweight='bold', inherit_aes=False)
    + scale_shape_manual(values=shape_vals, name=SIG_LEGEND)
    + scale_fill_manual(values=fill_vals, name=SIG_LEGEND)
    + labs(x=f'ProteinGym: {ci_axis} ({n_units_pg} {unit_word})',
           y=f'UKBBGym: {ci_axis} ({n_ukbb} genes)')
    + theme_minimal()
    + theme(figure_size=(6, 6), legend_position='bottom', legend_box_spacing=0.01,
            legend_margin=0, legend_direction='horizontal', legend_text=element_text(size=11),
            legend_title=element_text(size=11),
            axis_title=element_text(size=12, lineheight=1.4), strip_text=element_text(size=12))
)
suffix = '_genelevel' if UNIT_LEVEL == 'gene' else '_assaylevel'
# plot_overall.save(f'{FIG_DIR}/F4_SNR_pctileboot_UKBGym_v_proteingym_overall{suffix}.svg', dpi=200, verbose=False)
plot_overall

## Plot 1b — |signal-to-noise|, sign-agreement categories

Same predictor pairs as Plot 1, but the axes are magnitude only (`|SNR|`) and the state carries
five values instead of four: sign agreement is only meaningful once both datasets call a pair
significant, so `both` splits into **signs agree** / **signs disagree**; `ProteinGym only`,
`UKBBGym only`, and `neither` are unchanged from Plot 1's `significance` column, just recomputed
here as `significance5`.

In [ ]:
from matplotlib import colormaps
from matplotlib.colors import to_hex

CMAP5 = 'magma'
cmap5 = colormaps[CMAP5]
sig5_positions = {
    'Both, signs disagree': 0.0,
    'UKBBGym only':         0.25,
    'Neither':              0.45,
    'ProteinGym only':      0.75,
    'Both, signs agree':    0.95,
}
fill_vals5  = {k: to_hex(cmap5(v)) for k, v in sig5_positions.items()}
SIG5_LEGEND = 'Significant in'

pairs_overall_abs = pairs_overall.with_columns(pg_snr_abs=pl.col('pg_snr').abs(),
                                               ukbb_snr_abs=pl.col('ukbb_snr').abs())

abs_ci_axis = ci_axis.replace('Δcorr', 'mean |Δcorr|')

rho_overall_abs = pairs_overall_abs.select(
    spearman_r=pl.corr(pl.col('pg_snr_abs').rank('average'), pl.col('ukbb_snr_abs').rank('average')),
    x=pl.col('pg_snr_abs').min(), y=pl.col('ukbb_snr_abs').max(),
).with_columns(corr_label='ρ = ' + pl.col('spearman_r').round(2).cast(pl.Utf8))

plot_overall_abs = (
    ggplot(pairs_overall_abs, aes(x='pg_snr_abs', y='ukbb_snr_abs'))
    + geom_hline(yintercept=0.5, linetype='dotted', color='grey')
    + geom_vline(xintercept=0.5, linetype='dotted', color='grey')
    + geom_abline(slope=1, intercept=0, linetype='dashed', color='grey')
    + geom_point(aes(fill='significance5'), color='black', size=3.5,
                 stroke=0.4, alpha=0.85)
    + geom_text(data=rho_overall_abs, mapping=aes(x='x', y='y', label='corr_label'),
                ha='left', va='top', size=12, fontweight='bold', inherit_aes=False)
    + scale_fill_manual(values=fill_vals5, name=SIG5_LEGEND)
    + labs(x=f'ProteinGym: {abs_ci_axis} ({n_units_pg} {unit_word})',
           y=f'UKBBGym: {abs_ci_axis} ({n_ukbb} genes)')
    + guides(fill=guide_legend(nrow=2, byrow=True))
    + theme_minimal()
    + theme(
        figure_size=(6, 6), 
        legend_position='bottom', 
        legend_margin=0, 
        legend_direction='horizontal', 
        legend_text=element_text(size=12),
        legend_title=element_text(size=12),
        axis_title=element_text(size=12, lineheight=1.4), 
        axis_text=element_text(size=12),
        strip_text=element_text(size=12),
        panel_grid_major=element_line(color="#cccccc", size=0.6),
        panel_grid_minor=element_line(color="#dddddd", size=0.3),
    )
)
plot_overall_abs.save(f'{FIG_DIR}/F5_absSNR_pctileboot_UKBGym_v_proteingym_overall{suffix}.svg', dpi=200, verbose=False)
plot_overall_abs

In [ ]:
pairs_overall_abs.filter(pl.col('significance5') == 'Both, signs disagree')

### Bar plot — pairs by significance state

Counts of predictor pairs in each of the five `significance5` states above, same colors and order.

In [ ]:
sig5_counts = pairs_overall['significance5'].value_counts().sort('count', descending=True)
SIG5_CATEGORIES = list(sig5_counts['significance5'])
sig5_counts = sig5_counts.with_columns(significance5=pl.col('significance5').cast(pl.Enum(SIG5_CATEGORIES)))

plot_sig5_bar = (
    ggplot(sig5_counts, aes(x='significance5', y='count', fill='significance5'))
    + geom_col(width=0.8, alpha=0.9, show_legend=False)
    + geom_text(aes(label='count'), ha='center', size=12, nudge_y=1)
    + scale_fill_manual(values=fill_vals5)
    + labs(x='', y=f'predictor pairs (n={pairs_overall.height})')
    + theme_minimal()
    + theme(
        figure_size=(3.5, 5), 
        axis_text_x=element_text(size=12, angle=45, ha='right', va='top'),
        axis_title=element_text(size=12)
    )
)
plot_sig5_bar.save(f'{FIG_DIR}/F5_significance5_barplot{suffix}.svg', dpi=200, verbose=False)
plot_sig5_bar

### Contingency table — overall SNR significance

`significance` (cell 13) is already these same four counts as a flat value-count; this is the
2×2 it collapses, `ukbb_is_significant` against `pg_is_significant`, with row/column totals. The
diagonal (`both` / `neither`) is where the two datasets agree on whether a pair is resolvable at
all; the off-diagonal is where one CI excludes zero and the other doesn't. Agreement here says
nothing about direction -- a pair in the `both` cell can still rank the predictors oppositely,
that question is what the scatter's diagonal line and ρ are for.

In [ ]:
n_both      = int((pairs_overall['pg_is_significant'] & pairs_overall['ukbb_is_significant']).sum())
n_pg_only   = int((pairs_overall['pg_is_significant'] & ~pairs_overall['ukbb_is_significant']).sum())
n_ukbb_only = int((~pairs_overall['pg_is_significant'] & pairs_overall['ukbb_is_significant']).sum())
n_neither   = int((~pairs_overall['pg_is_significant'] & ~pairs_overall['ukbb_is_significant']).sum())
total = pairs_overall.height
assert n_both + n_pg_only + n_ukbb_only + n_neither == total

contingency = pl.DataFrame({
    '': ['UKBBGym significant', 'UKBBGym not significant', 'Total'],
    'ProteinGym significant':     [n_both, n_pg_only, n_both + n_pg_only],
    'ProteinGym not significant': [n_ukbb_only, n_neither, n_ukbb_only + n_neither],
    'Total':                      [n_both + n_ukbb_only, n_pg_only + n_neither, total],
})

agree = (n_both + n_neither) / total
print(f'agreement (both or neither significant): {n_both + n_neither} / {total} = {agree:.1%}')
print(f'UKBBGym significant, ProteinGym not:  {n_ukbb_only}')
print(f'ProteinGym significant, UKBBGym not:  {n_pg_only}')
contingency

In [ ]:
pg_only = (pairs_overall
           .filter(pl.col('pg_is_significant') & ~pl.col('ukbb_is_significant'))
           .select(['pair_label', 'category_a', 'pg_snr', 'pg_ci_lo', 'pg_ci_hi',
                    'ukbb_snr', 'ukbb_ci_lo', 'ukbb_ci_hi'])
           .sort('pg_snr', descending=True))
print(f'{pg_only.height} pairs: significant in ProteinGym, not in UKBBGym')
pg_only